In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report
import os

In [2]:
import numpy as np
import pandas as pd
import os

base_path = "/content/drive/MyDrive/Colab Notebooks/processed_features/pump"

id_folders = ['id_00', 'id_02', 'id_04', 'id_06']

datasets = {}
for id_folder in id_folders:
    records = []

    for label in ['normal', 'abnormal']:
        folder_path = os.path.join(base_path, id_folder, label)

        for file in sorted(os.listdir(folder_path)):
            if file.endswith(".npy"):
                file_path = os.path.join(folder_path, file)
                data = np.load(file_path, allow_pickle=True).flatten()

                row = {f'col_{i}': val for i, val in enumerate(data)}
                row['label'] = int(id_folder[-1])/2

                records.append(row)

    datasets[id_folder] = pd.DataFrame(records)
    print(f"{id_folder} -> shape: {datasets[id_folder].shape}")



id_00 -> shape: (1149, 9)
id_02 -> shape: (1116, 9)
id_04 -> shape: (802, 9)
id_06 -> shape: (1138, 9)


In [3]:
id_00 = datasets['id_00']
id_02 = datasets['id_02']
id_04 = datasets['id_04']
id_06 = datasets['id_06']

In [4]:
data = pd.concat([id_00,id_04,id_02,id_06], ignore_index=True)

In [5]:
data.info()
data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4205 entries, 0 to 4204
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   col_0   4205 non-null   float32
 1   col_1   4205 non-null   float32
 2   col_2   4205 non-null   float32
 3   col_3   4205 non-null   float32
 4   col_4   4205 non-null   float32
 5   col_5   4205 non-null   float32
 6   col_6   4205 non-null   float32
 7   col_7   4205 non-null   float32
 8   label   4205 non-null   float64
dtypes: float32(8), float64(1)
memory usage: 164.4 KB


,col_0,col_1,col_2,col_3,col_4,col_5,col_6,col_7,label
0,0.308204,0.000342,9.583734e-08,1134.645874,2728.548096,4791.683105,-738.275391,42.612053,0.0
1,0.219160,0.000410,1.353300e-07,1154.479126,2148.921143,3751.597412,-747.351013,74.212547,0.0
2,0.231815,0.000434,1.643607e-07,1171.276245,2279.358887,4011.456787,-753.793701,72.428001,0.0
3,0.205733,0.000430,1.429113e-07,1128.110840,2119.113770,3663.837891,-750.353027,68.951477,0.0
4,0.215199,0.000437,1.752913e-07,1141.218750,2202.336182,3700.803711,-744.250793,66.060593,0.0


In [6]:
import torch.nn as nn
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.input     = nn.Linear(8, 64)
        self.bn1       = nn.BatchNorm1d(64)
        self.dropout1  = nn.Dropout(0.3)

        self.fc1       = nn.Linear(64, 128)
        self.bn2       = nn.BatchNorm1d(128)
        self.dropout2  = nn.Dropout(0.3)

        self.fc2       = nn.Linear(128, 64)
        self.bn3       = nn.BatchNorm1d(64)
        self.dropout3  = nn.Dropout(0.2)

        self.output    = nn.Linear(64, 4)

        self.optimizer = torch.optim.Adam(self.parameters(), lr=0.001, weight_decay=1e-4)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, patience=20, factor=0.5)
        self.loss_fun  = nn.CrossEntropyLoss()

    def forward(self, x):
        x = F.relu(self.bn1(self.input(x)))
        x = self.dropout1(x)
        x = F.relu(self.bn2(self.fc1(x)))
        x = self.dropout2(x)
        x = F.relu(self.bn3(self.fc2(x)))
        x = self.dropout3(x)
        return self.output(x)

    def train_model(self, X, y):

        numepochs = 2500
        X_tensor = torch.tensor(X.values, dtype=torch.float32)
        y_tensor = torch.tensor(y.values, dtype=torch.long)

        train_data_tensor, test_data_tensor, train_labels_tensor, test_labels_tensor = train_test_split(
            X_tensor, y_tensor, test_size=0.1, random_state=42, stratify=y_tensor
        )

        train_dataset = TensorDataset(train_data_tensor, train_labels_tensor)
        test_dataset  = TensorDataset(test_data_tensor,  test_labels_tensor)

        batchsize    = 64
        train_loader = DataLoader(train_dataset, batch_size=batchsize, shuffle=True, drop_last=True)
        test_loader  = DataLoader(test_dataset,  batch_size=test_dataset.tensors[0].shape[0])

        losses   = torch.zeros(numepochs)
        trainAcc = []
        testAcc  = []

        best_test_acc = 0
        best_weights  = None

        for epochi in range(numepochs):
            self.train()
            batchAcc  = []
            batchLoss = []

            for X_batch, y_batch in train_loader:
                yHat = self(X_batch)
                loss = self.loss_fun(yHat, y_batch)

                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                batchLoss.append(loss.item())
                matches        = torch.argmax(yHat, axis=1) == y_batch
                accuracyPct    = 100 * torch.mean(matches.float())
                batchAcc.append(accuracyPct)

            trainAcc.append(np.mean(batchAcc))
            losses[epochi] = np.mean(batchLoss)


            self.eval()
            X_test, y_test = next(iter(test_loader))
            with torch.no_grad():
                yHat = self(X_test)

            acc = 100 * torch.mean((torch.argmax(yHat, axis=1) == y_test).float())
            testAcc.append(acc)

            self.scheduler.step(losses[epochi])
            if acc > best_test_acc:
                best_test_acc = acc
                best_weights  = {k: v.clone() for k, v in self.state_dict().items()}

            if (epochi + 1) % 100 == 0:
                print(f"Epoch {epochi+1}/{numepochs} | Loss: {losses[epochi]:.4f} | Train: {trainAcc[-1]:.2f}% | Test: {acc:.2f}%")

        self.load_state_dict(best_weights)
        print(f"\nBest Test Accuracy: {best_test_acc:.2f}%")

        return trainAcc, testAcc, losses

In [9]:
X = data.drop('label', axis=1)
y = data['label']

model = Model()

In [10]:
model.train_model(X,y)

Epoch 100/2500 | Loss: 0.6452 | Train: 75.58% | Test: 80.76%
Epoch 200/2500 | Loss: 0.6149 | Train: 76.99% | Test: 79.10%
Epoch 300/2500 | Loss: 0.5852 | Train: 78.47% | Test: 79.33%
Epoch 400/2500 | Loss: 0.5860 | Train: 77.70% | Test: 79.81%
Epoch 500/2500 | Loss: 0.6012 | Train: 76.75% | Test: 81.95%
Epoch 600/2500 | Loss: 0.5959 | Train: 77.65% | Test: 81.47%
Epoch 700/2500 | Loss: 0.5793 | Train: 77.54% | Test: 80.29%
Epoch 800/2500 | Loss: 0.5852 | Train: 77.73% | Test: 79.81%
Epoch 900/2500 | Loss: 0.5873 | Train: 77.73% | Test: 81.00%
Epoch 1000/2500 | Loss: 0.5910 | Train: 77.60% | Test: 79.10%
Epoch 1100/2500 | Loss: 0.5936 | Train: 77.73% | Test: 81.00%
Epoch 1200/2500 | Loss: 0.5897 | Train: 77.89% | Test: 81.00%
Epoch 1300/2500 | Loss: 0.5912 | Train: 77.81% | Test: 80.76%
Epoch 1400/2500 | Loss: 0.5962 | Train: 77.54% | Test: 80.52%
Epoch 1500/2500 | Loss: 0.5911 | Train: 77.20% | Test: 81.00%
Epoch 1600/2500 | Loss: 0.5921 | Train: 77.60% | Test: 80.29%
Epoch 1700/2500 |

([np.float32(51.800846),
  np.float32(61.04343),
  np.float32(65.43962),
  np.float32(67.87606),
  np.float32(68.00848),
  np.float32(70.4714),
  np.float32(70.04767),
  np.float32(70.94809),
  np.float32(71.504234),
  np.float32(70.60381),
  np.float32(71.58369),
  np.float32(71.769066),
  np.float32(71.742584),
  np.float32(71.58369),
  np.float32(71.90148),
  np.float32(71.7161),
  np.float32(71.7161),
  np.float32(72.29873),
  np.float32(73.119705),
  np.float32(72.93432),
  np.float32(73.04025),
  np.float32(73.04025),
  np.float32(73.35805),
  np.float32(72.643005),
  np.float32(73.38454),
  np.float32(73.305084),
  np.float32(72.93432),
  np.float32(73.19915),
  np.float32(73.808266),
  np.float32(72.98729),
  np.float32(73.46398),
  np.float32(74.17902),
  np.float32(74.52331),
  np.float32(74.15254),
  np.float32(74.33792),
  np.float32(74.73517),
  np.float32(74.443855),
  np.float32(73.38454),
  np.float32(74.47034),
  np.float32(74.54979),
  np.float32(74.28496),
  np.float

In [ ]:
from joblib import dump
dump(model, 'pump_model.joblib')